# APM_50441_EP - Machine Learning 2 - Lab 3 (companion notebook): AutoGluon

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APM-50441-EP/APM_50441_EP-2026-2027/blob/main/PC3/APM_50441_EP_Lab_3_AutoGluon.ipynb) or run it on your own computer (see *Working environment* below).

This short notebook complements the *To go further* part of Lab 3 with AutoGluon, an AutoML framework for tabular data. Its goals are:
- to run AutoGluon with a small time budget, and to read its leaderboard,
- to understand what it does inside: bagging, stacking and weighted ensembles,
- to use tabular foundation models (TabICLv2, TabDPT) through AutoGluon,
- to compare all these models with models of Lab 3, on the cost-based gain of Lab 1.

It takes about $20$ minutes with a GPU, and longer on a CPU.

We use `Python` with `autogluon.tabular`, `scikit-learn` and `polars`.

**Working environment.** Colab is the simplest option: open this notebook from the course repository (*Open in Colab* link), choose a GPU runtime (*Runtime > Change runtime type*) and run it. To work on your own computer, download the folder of Lab 3 (this notebook, the data and `apm_credit.py`) or clone the course repository, and use a *separate* environment, as AutoGluon pins many packages: create it once with [`uv`](https://docs.astral.sh/uv/) (`uv venv .venv-autogluon`, activate it with `source .venv-autogluon/bin/activate`, or `.venv-autogluon\Scripts\activate` on Windows, and `uv pip install jupyter`), then select it as the kernel of the notebook in your IDE (VS Code with the Python and Jupyter extensions, JupyterLab, Spyder, PyCharm). The installation cell below installs AutoGluon into the Python running the notebook; with an NVIDIA GPU, add `--torch-backend=auto` to it to get the CUDA version of PyTorch. TabDPT needs a GPU; the other models also run on a CPU, more slowly.

**Use of AI.** AI assistants are allowed, but you must be able to explain every line you use, and you should answer the *predict before you run* questions before running the code: the suggested code may not be correct!

AutoGluon and its dependencies are installed at once (a few minutes). On Colab, restart the session once after this installation; locally, restart the kernel if the installation upgraded a package that was already loaded.

In [ ]:
# Installs into the Python running this notebook: Colab, or locally an activated virtual environment
import sys
!uv pip install --python "{sys.executable}" "autogluon.tabular[all,tabicl,tabdpt]==1.6.2" "scikit-learn==1.9.1" "polars==1.44.2" "seaborn==0.13.2" "fairlearn==0.14.0"

We use the German credit data and the cost model of Lab 1 (`apm_credit.py`), with the same split as Lab 3. Each file is downloaded only if there is no local copy.

In [ ]:
import os
import urllib.request

# each file is downloaded only if there is no local copy
CSV_URL = "https://raw.githubusercontent.com/IBM/monitor-wml-model-with-watson-openscale/master/data/german_credit_data_biased_training.csv"
APM_CREDIT_URL = "https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/common/apm_credit.py"  # course repository
for url in [CSV_URL, APM_CREDIT_URL]:
  filename = url.split('/')[-1]
  if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

In [ ]:
import time, tempfile, warnings
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn.objects as so
from sklearn.model_selection import train_test_split
import apm_credit
from apm_credit import cost_metric, cost_predict_from_single_proba
warnings.filterwarnings('ignore', message='IProgress not found')  # tqdm without ipywidgets

german_credit = pl.read_csv("german_credit_data_biased_training.csv")
X = german_credit.drop('Risk')
y = german_credit['Risk']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
LoanAmount_test = X_test['LoanAmount']
apm_credit.set_proportions(y_train, {'Risk': .02, 'No Risk': .98})
train_data = X_train.with_columns(y_train).to_pandas()  # AutoGluon works on pandas data frames, with the label
test_data = X_test.to_pandas()
test_probas, fit_times = {}, {}  # test probabilities of Risk and fit times of the models, for the final summary

def test_gain(proba_risk):
  # real-world gain of the Bayes decision computed from the probabilities of Risk (Lab 1)
  return -cost_metric(y_test, cost_predict_from_single_proba(np.asarray(proba_risk), LoanAmount_test), LoanAmount_test)

As references, we fit two models of Lab 3 on the same data: a logistic regression and a histogram gradient boosting, with the preprocessing of Lab 3.

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
preprocessor = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), X.select(cs.string()).columns),
                                  ('num', StandardScaler(), X.select(cs.numeric()).columns)])
for name, model in {'logistic regression': LogisticRegression(max_iter=1000),
                    'gradient boosting': HistGradientBoostingClassifier(random_state=0)}.items():
  start = time.perf_counter()
  fitted = make_pipeline(clone(preprocessor), model).fit(X_train, y_train)
  fit_times[name] = time.perf_counter() - start
  test_probas[name] = fitted.predict_proba(X_test)[:, list(fitted.classes_).index('Risk')]
{name: round(test_gain(proba), 2) for name, proba in test_probas.items()}

## 1. AutoGluon in three minutes

`TabularPredictor` takes a data frame with the label, an evaluation metric and a time budget. With the preset `medium_quality`, it holds out a validation set, trains several kinds of models one after the other (LightGBM, CatBoost, XGBoost, random forests, extra trees, neural networks...) until the budget is spent, and combines them in a weighted ensemble. We use the log loss, as our decision relies on the probabilities. The models are written in a temporary folder.

**Question** Run `TabularPredictor(label='Risk', eval_metric='log_loss')` with `presets='medium_quality'` and `time_limit=180`, compute the test gain of the Bayes decision on its probabilities, and compare it with the two reference models. Look at its leaderboard: which models were trained, how long did they take, which one is the best on the validation set?

**Answer**

---

## 2. Inside AutoGluon: bagging and stacking

With `num_bag_folds=k`, each model is *bagged*: it is trained $k$ times on $k-1$ folds and predicts the remaining fold, which gives out-of-fold predictions on the whole training set (as in a cross validation), and the $k$ models are averaged. With `num_stack_levels=1`, a second layer of models is trained on the original features *and* the out-of-fold predictions of the first layer: this is *stacking* (Lecture 3, and the `StackingClassifier` of Lab 3). The final `WeightedEnsemble` averages the best models, with weights chosen greedily on the out-of-fold predictions (ensemble selection).

**Question** Fit AutoGluon again with `num_bag_folds=5`, `num_stack_levels=1`, `dynamic_stacking=False` and `time_limit=300`. In the leaderboard, identify the models of each layer (`stack_level`) and the final ensemble, and print the weights of the ensemble (with the internal function `predictor._trainer.load_model(predictor.model_best)._get_model_weights()`). Is the test gain better?

**Answer**

---

## 3. Tabular foundation models through AutoGluon

TabICLv2 and TabDPT are pretrained transformers: they do not fit parameters on our data, but use the training set as a *context* to predict the test points (in-context learning, Lecture 3). AutoGluon gives access to them with `hyperparameters={'TABICL': {}}` or `{'TABDPT': {}}`. With `refit_full=True`, the final model uses the whole training set as context, instead of the part left after the validation split. TabDPT needs a GPU; TabICLv2 also runs on a CPU, more slowly.

**Question** Fit TabICLv2 and TabDPT through AutoGluon (`time_limit=600`, `refit_full=True`, `set_best_to_refit_full=True`), and compare their test gains and their fit and prediction times with the other models.

**Answer**

---

## 4. Summary

All the models are evaluated on the same $1000$ test loans. To see which differences are meaningful, we compute a $95\%$ bootstrap confidence interval of each gain, and of its difference with the best model (the same resamples are used for all the models).

**Question** Compute these intervals, plot them with the fit times, and conclude: which differences are significant? What does AutoGluon cost?

**Answer**

---

## About this lab

**Use of AI.** Artificial intelligence tools have been used to revise the content of this lab. The main contributor takes full responsibility for this content.

**License.** The texts of this lab are under the [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license (CC BY-SA 4.0), and its code under the [MIT](https://opensource.org/license/mit) license; the datasets keep their own licenses. Main contributor: E. Le Pennec. Contributors: S. Boucheron, A. Dieuleveut, A.K. Fermin, S. Gadat, S. Gaiffas, B. Goujaud, A. Guilloux, Ch. Keribin, Th. Klein, E. Matzner, J. Read, M. Sangnier, E. Scornet.